Hello, my name is Aaron Kann, and this is an analysis on soccer stats.  The goal of this first analysis is to determine whether,
in general, a players last 5 stats are a better predictor of the next match's output as compared to the season to date.

In [1]:
import sys
import os

sys.path.append('..')  # Add parent folder to the module search path
import fbref_scrape as fbref
from constants import *


Firstly, the data for the premier league can be scraped through my code.  It will take a long time, so only uncomment the code if absolutely necessary.
The data should be stored in a folder called Premier_League_Matchlogs and accessed through there, so unless you are updating the folder, it is unnessecary to run the following code strip.

In [2]:
import json, csv, os
import pandas as pd
import numpy as np
from scipy.stats import poisson

# uncommenting the following line will scrape the premier league data
# fbref.SoccerLeague(fbref.premier_league_24_url, {
#     'list_teams': [16],
#     'file_location': '..'
# })

Next, we need to take the indiviual player's gamelogs and create a giant database of games.  
Because the variables we are trying to measure aren't explicitly within the data, we need to calculate them.

In [4]:
# where am i?
print(os.getcwd())
# go back a folder
if os.getcwd().endswith('data_analyses'):
    os.chdir('..')
print(os.getcwd())

c:\Users\Aaron\OneDrive\Documents\GitHub\soccer-stat-data-analysis\data_analyses
c:\Users\Aaron\OneDrive\Documents\GitHub\soccer-stat-data-analysis


In [5]:
# read in the data
team_df = pd.read_csv('2023_2024_Premier_League_Team_Stats.csv')

data_folder = 'Premier_League_Player_Matchlogs'
data_files = os.listdir(data_folder)
dfs = []

for file in data_files:
    if file.endswith('.csv'):
        file_path = os.path.join(data_folder, file)
        # read the csv file and process the data
        df = pd.read_csv(file_path)
        # df = add_name_to_df(df, file)
        dfs.append(df)

dfs[0].head()


,game_started,game_count,dayofweek,comp,round,venue,result,team,opponent,position,...,gca,passes_completed,passes,passes_pct,progressive_passes,carries,progressive_carries,take_ons,take_ons_won,match_report
0,Y,1,Mon,Premier League,Matchweek 1,Home,W 1-0,Manchester Utd,Wolves,RB,...,1,47,51,92.2,3,42,1,4,3,Match Report
1,Y,2,Sat,Premier League,Matchweek 2,Away,L 0-2,Manchester Utd,Tottenham,RB,...,0,38,44,86.4,1,29,1,2,1,Match Report
2,Y,3,Sat,Premier League,Matchweek 3,Home,W 3-2,Manchester Utd,Nott'ham Forest,RB,...,0,58,69,84.1,9,55,7,2,1,Match Report
3,Y,4,Sun,Premier League,Matchweek 4,Away,L 1-3,Manchester Utd,Arsenal,RB,...,0,44,54,81.5,2,30,2,0,0,Match Report
4,N,5,Sat,Premier League,Matchweek 5,Home,L 1-3,Manchester Utd,Brighton,CB,...,0,10,11,90.9,1,5,0,0,0,Match Report


In [21]:
# def add_name_to_df(df, filename):
#     for row in df

def update_avgs(season_avgs, row, index):
    #iterate through the stats in season_avgs and update the averages
    
    for stat in season_avgs.keys():
        if stat in row.keys():
            season_avgs[stat] = (float(season_avgs[stat])*index + float(row[stat]))/(index+1)
    return season_avgs

def update_last_five_avgs(last_five_avgs, df, index):
    #iterate through the stats in last_five_avgs and update the averages
    for stat in last_five_avgs.keys():
        if stat in df.keys():
            last_five_avgs[stat] = (last_five_avgs[stat]*5 + df.iloc[index][stat] - df.iloc[index-5][stat])/(5)
    return last_five_avgs

In [23]:
stats_wanted = fbref.get_stats_wanted('fbref')
stats_wanted.remove('name')
stats_wanted.remove('team')
stats_wanted.append('game_count')
df_as_list = list()

for i in range(len(dfs)): #iterates through each player's data
    
    # create a new row, with the season totals as of matchweek 0 (ie all zero)
    season_avgs = pd.Series(data=[0.0]*len(stats_wanted), index=stats_wanted)
    last_five_avgs = pd.Series(data=[0.0]*len(stats_wanted), index=stats_wanted)
    # print(season_avgs, last_five_avgs)

    for index, row in dfs[i].iterrows(): #iterates through each game of each players data
        dfrow = pd.concat([row, season_avgs.add_suffix("_szn"), last_five_avgs.add_suffix("_l5")], axis=0)

        #TODO: scale this to all stats
        dfrow['passes_l5_minus_szn'] = dfrow['passes_l5'] - dfrow['passes_szn']
        dfrow['opp_passes_against'] = team_df[team_df['name'] == row['opponent']]['passes_against'].values[0] / team_df['passes_against'].mean()


        season_avgs = update_avgs(season_avgs, row, index)
        if index >= 5:
            last_five_avgs = update_last_five_avgs(last_five_avgs, dfs[i], index)
        else:
            last_five_avgs = update_avgs(last_five_avgs, row, index)

        df_as_list.append(dfrow)

    #end game loop
#end player loop
        


df = pd.DataFrame(df_as_list)
df.head()

,game_started,game_count,dayofweek,comp,round,venue,result,team,opponent,position,...,game_count_l5,passes_l5_minus_szn,opp_passes_against,games,gk_saves,clearances,assisted_shots,fouled,crosses,fouls
0,Y,1,Mon,Premier League,Matchweek 1,Home,W 1-0,Manchester Utd,Wolves,RB,...,0.0,0.0,1.041747,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Y,2,Sat,Premier League,Matchweek 2,Away,L 0-2,Manchester Utd,Tottenham,RB,...,1.0,0.0,0.776799,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Y,3,Sat,Premier League,Matchweek 3,Home,W 3-2,Manchester Utd,Nott'ham Forest,RB,...,1.5,0.0,1.184296,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Y,4,Sun,Premier League,Matchweek 4,Away,L 1-3,Manchester Utd,Arsenal,RB,...,2.0,0.0,0.834888,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,N,5,Sat,Premier League,Matchweek 5,Home,L 1-3,Manchester Utd,Brighton,CB,...,2.5,0.0,0.861054,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df.shape

#save df as a csv file
df.to_csv('output.csv', index=False)


In [16]:
import statsmodels.api as sm

df = df.dropna(subset=['passes_szn', 'passes'])

# Sample data
X = np.array(df['passes_szn'])
Y = np.array(df['passes'])

# Add a constant to the independent variable for the intercept
X = sm.add_constant(X)

# Fit the regression model
model = sm.OLS(Y, X).fit()


# Calculate the residual standard deviation
print(model.summary())
print("Residual Standard Deviation:", np.sqrt(model.scale)) 


                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.389
Model:                            OLS   Adj. R-squared:                  0.389
Method:                 Least Squares   F-statistic:                     4658.
Date:                Thu, 22 Jan 2026   Prob (F-statistic):               0.00
Time:                        13:07:13   Log-Likelihood:                -32272.
No. Observations:                7304   AIC:                         6.455e+04
Df Residuals:                    7302   BIC:                         6.456e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         10.1190      0.465     21.747      0.0

In [17]:
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.389
Model:                            OLS   Adj. R-squared:                  0.389
Method:                 Least Squares   F-statistic:                     4658.
Date:                Thu, 22 Jan 2026   Prob (F-statistic):               0.00
Time:                        13:07:20   Log-Likelihood:                -32272.
No. Observations:                7304   AIC:                         6.455e+04
Df Residuals:                    7302   BIC:                         6.456e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         10.1190      0.465     21.747      0.000       9.207      11.031
x1             0.7671      0.011     68.252      0.000       0.745       0.789
==============================================================================
Omnibus:                     1040.625   Durbin-Watson:                   1.739
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             2427.429
Skew:                           0.827   Prob(JB):                         0.00
Kurtosis:                       5.289   Cond. No.                         82.0
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

This shows a standard linear regression between a player's season to date total for passes attempted, and the resulting passes a player attempted in the game.
While the two variables are positively correlated (a r^2 value of 4/9 shows this, as well as a p value of zero), the high residual standard deviation shows that season averages alone isn't a great predictor of a player's single game output. 

Next, we will try adding a player's last 5 results to the regression, to see if this improves the accuracy of the model.

In [18]:
X = np.array(df[['passes_szn','passes_l5']])
X = sm.add_constant(X)
model = sm.OLS(Y, X).fit()

model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.396
Model:                            OLS   Adj. R-squared:                  0.396
Method:                 Least Squares   F-statistic:                     2397.
Date:                Thu, 22 Jan 2026   Prob (F-statistic):               0.00
Time:                        13:07:22   Log-Likelihood:                -32230.
No. Observations:                7304   AIC:                         6.447e+04
Df Residuals:                    7301   BIC:                         6.449e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          9.8786      0.463     21.314      0.000       8.970      10.787
x1             0.4671      0.035     13.405      0.000       0.399       0.535
x2             0.3028      0.033      9.091      0.000       0.238       0.368
==============================================================================
Omnibus:                     1079.024   Durbin-Watson:                   1.811
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             2589.982
Skew:                           0.845   Prob(JB):                         0.00
Kurtosis:                       5.378   Cond. No.                         117.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [19]:
print("Residual Standard Deviation:", np.sqrt(model.scale))

Residual Standard Deviation: 19.965010147496532


The improvement in the regression shows that using a player's last 5 games as data slightly improves the predictive power of the model. 

Note that, as season averages has a greater slope than last 5 averages, it can be concluded that season averages are a slightly better predictor of future outcomes than a player's last 5 averages, however the best way to predict future outcomes is to consider both averages. 

Next, let's try factoring in the opponent's averages against, and see if this improves the model more.  Personally, I think this is the second biggest factor in determining output, so I am expecting a large decrease in residential standard deviation. 

In [20]:
X = np.array(df[['passes_szn','passes_l5_minus_szn', 'opp_passes_against']])
X = sm.add_constant(X)
model = sm.OLS(Y, X).fit()

print("Residual Standard Deviation:", np.sqrt(model.scale))
model.summary()

Residual Standard Deviation: 19.002008283866402


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.453
Model:                            OLS   Adj. R-squared:                  0.453
Method:                 Least Squares   F-statistic:                     2017.
Date:                Thu, 22 Jan 2026   Prob (F-statistic):               0.00
Time:                        13:07:24   Log-Likelihood:                -31869.
No. Observations:                7304   AIC:                         6.375e+04
Df Residuals:                    7300   BIC:                         6.377e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        -30.3982      1.526    -19.916      0.000     -33.390     -27.406
x1             0.7661      0.011     71.980      0.000       0.745       0.787
x2             0.3069      0.032      9.681      0.000       0.245       0.369
x3            40.4543      1.468     27.564      0.000      37.577      43.331
==============================================================================
Omnibus:                      935.295   Durbin-Watson:                   1.770
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             2546.411
Skew:                           0.706   Prob(JB):                         0.00
Kurtosis:                       5.525   Cond. No.                         390.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

This provides an improvement than adding last 5 stats, however, there is still a large amount of deviance shown in the data, meaning more variables might be needed to predict it.  A good suggestion might be to show the expected standard deviation of a poisson process with means at the line of best fit, in order to provide a target standard deviation in which to aim for.